# WIMHF PRISM Quickstart

This notebook walks through the WIMHF pipeline step-by-step, starting from the PRISM subset hosted on Hugging Face:
https://huggingface.co/datasets/rmovva/wimhf-data.

We demonstrate how to:
1. Load and preprocess a preference dataset
2. Get embeddings and evaluate their predictive power
3. Train a sparse autoencoder (SAE)
4. Interpret SAE features using LLMs
5. Score interpretation fidelity
6. Compute controlled logit coefficients

The final step produces a feature table with interpretations, fidelity scores, and coefficients.


## Setup

Prerequisites:
- Install the WIMHF package in editable mode
- Export your OpenAI-compatible API key as an environment variable `OAI_WIMHF`
- Ensure you can access Hugging Face datasets (the config below pulls PRISM via HTTPS)


In [1]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from wimhf.quickstart import (
    DatasetConfig,
    EmbeddingConfig,
    SAEConfig,
    InterpretationSettings,
    SelectionSettings,
    RuntimeConfig,
    WIMHFConfig,
    load_and_preprocess_dataframe,
    get_embeddings,
    train_sae,
    interpret_sae_features,
    score_interpretations,
    compute_controlled_logit_coefs,
)
from wimhf.llm_tasks import abbreviate_concept, parallel_apply
from wimhf.utils import measure_interpretation_redundancy, fit_linear_model

# Set GPU
if "CUDA_VISIBLE_DEVICES" not in os.environ:
    os.environ["CUDA_VISIBLE_DEVICES"] = "1"


## Step 1: Load and Preprocess Dataframe

Load the dataset and split into train/validation sets using connected components to avoid leakage.


In [11]:
dataset_cfg = DatasetConfig(
    name="PRISM",
    path="https://huggingface.co/datasets/rmovva/wimhf-data/resolve/main/data/prism.json",
    split_columns=["prompt", "conversation_id"],
    train_split_size=0.8,
    split_random_seed=42,
    prompt_column="prompt",
    response_a_column="response_A",
    response_b_column="response_B",
    label_column="label",
    max_words_prompt=128,
    max_words_response=256,
    random_seed=42,
)

runtime_cfg = RuntimeConfig(
    checkpoint_dir="outputs/prism/checkpoints",
    cache_dir=Path("outputs/prism/emb_cache"),
    retrain_sae=True,
)

train_df, val_df, train_pred_df, val_pred_df, dedup_train_df, dedup_val_df = load_and_preprocess_dataframe(
    dataset_cfg, runtime_cfg
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}")
print(f"Binary labels - Train: {len(train_pred_df)}, Val: {len(val_pred_df)}")


Building graph edges...

Finding connected components...
Connected-component split -> train 20738 (80.0%), val 5180 (20.0%)
[dedup train] 20738 -> 20636 rows (unique unordered pairs)
[dedup val] 5180 -> 5159 rows (unique unordered pairs)
Train: 20738, Val: 5180
Binary labels - Train: 13884, Val: 3472


In [7]:
runtime_cfg = RuntimeConfig(
    checkpoint_dir="outputs/prism/checkpoints",
    cache_dir=Path("outputs/prism/emb_cache"),
    retrain_sae=True,
)

## Step 2: Get Response Embeddings, Compute AUC

Get embeddings for all responses and evaluate how well text embeddings predict the preference label.

Specifically, we fit a logistic regression to predict $y$ from $e_\Delta = e_{r_A} - e_{r_B}$.


In [12]:
embedding_cfg = EmbeddingConfig(
    model="text-embedding-3-small",
    n_workers=2,
)

response2embedding, delta_train, delta_val, dedup_delta_train, dedup_delta_val = get_embeddings(
    train_df,
    val_df,
    dedup_train_df,
    dedup_val_df,
    dataset_cfg,
    embedding_cfg,
    cache_dir=runtime_cfg.cache_dir,
)

Loading embedding cache:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
# Compute AUC with embedding deltas
train_pred_mask = train_df[dataset_cfg.label_column].isin([0, 1]).to_numpy()
val_pred_mask = val_df[dataset_cfg.label_column].isin([0, 1]).to_numpy()
delta_train_pred = delta_train[train_pred_mask]
delta_val_pred = delta_val[val_pred_mask]

control_train = train_pred_df[["length_delta"]].to_numpy()
control_val = val_pred_df[["length_delta"]].to_numpy()
y_train = train_pred_df[dataset_cfg.label_column].to_numpy()
y_val = val_pred_df[dataset_cfg.label_column].to_numpy()

X_train = np.concatenate([delta_train_pred, control_train], axis=1)
X_val = np.concatenate([delta_val_pred, control_val], axis=1)
embedding_train_auc, embedding_val_auc = fit_linear_model(
    X_train, y_train, X_val, y_val, is_binary=True, standardize=False,
)

print(f"Embedding AUC - Train: {embedding_train_auc:.3f}, Val: {embedding_val_auc:.3f}")

Embedding AUC - Train: 0.736, Val: 0.673


## Step 3: Train SAE

Train a sparse autoencoder to learn sparse representations of response differences.


In [14]:
sae_cfg = SAEConfig(
    M=32,
    K=4,
    prefix_lengths=[8, 32],
    batch_size=256,
    n_epochs=50,
    min_epochs=10,
    patience=3,
    learning_rate=5e-4,
    dead_neuron_threshold_steps=64,
)

sae = train_sae(
    dedup_delta_train,
    dedup_delta_val,
    sae_cfg,
    runtime_cfg,
)

print(f"SAE trained: {sae_cfg.M} total neurons, {sae_cfg.K} active per example")


  0%|          | 0/50 [00:00<?, ?it/s]

[sae] early stopping after 42 epochs
[sae] saved model to outputs/prism/checkpoints/SAE_matryoshka_M=32_K=4_prefixes=8-32.pt
SAE trained: 32 total neurons, 4 active per example


## Step 4: Compute SAE AUC

Get SAE activations and evaluate how well they predict preferences compared to raw embeddings.


In [15]:
activ_train = sae.get_activations(delta_train)
activ_val = sae.get_activations(delta_val)
activ_dedup = sae.get_activations(dedup_delta_train)

# Analyze SAE sparsity and dead features
n_dead_features = (activ_dedup.sum(axis=0) == 0).sum()
mean_active_per_input = (activ_dedup != 0).sum(axis=1).mean()

print(f"# SAE features that never activate (dead): {n_dead_features}/{sae_cfg.M}")
print(f"Mean active features per input: {mean_active_per_input:.2f}")

# Compute SAE AUC
train_pred_mask = train_df[dataset_cfg.label_column].isin([0, 1]).to_numpy()
val_pred_mask = val_df[dataset_cfg.label_column].isin([0, 1]).to_numpy()
activ_train_pred = activ_train[train_pred_mask]
activ_val_pred = activ_val[val_pred_mask]

control_train = train_pred_df[["length_delta"]].to_numpy()
control_val = val_pred_df[["length_delta"]].to_numpy()
y_train = train_pred_df[dataset_cfg.label_column].to_numpy()
y_val = val_pred_df[dataset_cfg.label_column].to_numpy()

X_train = np.concatenate([activ_train_pred, control_train], axis=1)
X_val = np.concatenate([activ_val_pred, control_val], axis=1)
sae_train_auc, sae_val_auc = fit_linear_model(
    X_train, y_train, X_val, y_val, is_binary=True, standardize=False
)

print(f"SAE AUC - Train: {sae_train_auc:.3f}, Val: {sae_val_auc:.3f}")


Computing activations (batch=16384):   0%|          | 0/2 [00:00<?, ?it/s]

Computing activations (batch=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batch=16384):   0%|          | 0/2 [00:00<?, ?it/s]

# SAE features that never activate (dead): 0/32
Mean active features per input: 4.00
SAE AUC - Train: 0.668, Val: 0.670


## Step 5: Interpret SAE Features

Use LLMs to interpret what each SAE neuron detects by analyzing examples where it activates.


In [16]:
interpret_cfg = InterpretationSettings(
    interpreter_model="gpt-5",
    annotator_model="gpt-5-mini",
    abbreviator_model="gpt-5-mini",
    n_candidates=5,
    interpret_n_examples=10,
    interpret_sampling_kwargs={"percentile_bin": (95, 100), "nonzero_only": True, "swap_negative_response_pairs": True},
    scoring_n_examples=200,
    scoring_sampling_kwargs={"high_percentile": (50, 100), "low_percentile": (0, 50), "nonzero_only": True},
    min_correlation=0.0,
    p_value_threshold=0.05 / 32,  # Bonferroni correction
    n_workers_interpretation=50,
    n_workers_annotation=300,
    timeout=120.0,
    max_interpretation_tokens=15000,
    completion_kwargs={"reasoning_effort": "low"},
)

interpretations, interpreter = interpret_sae_features(
    dedup_train_df,
    activ_dedup,
    dataset_cfg,
    interpret_cfg,
    runtime_cfg,
)

print(f"Generated interpretations for {len(interpretations)} neurons")


Generating 5 interpretation(s) per neuron:   0%|          | 0/160 [00:00<?, ?it/s]

Generated interpretations for 32 neurons


## Step 6: Score Interpretations

Evaluate how well each interpretation correlates with actual neuron activations (fidelity scoring).


In [17]:
scoring_metrics, best_interpretations, best_metrics, kept_neurons = score_interpretations(
    interpretations,
    train_df,
    activ_train,
    dataset_cfg,
    interpret_cfg,
    interpreter,
)

correlations = [abs(m.get("correlation", 0.0)) for m in best_metrics.values()]
mean_abs_corr = float(np.nanmean(correlations)) if correlations else float("nan")

print(f"Mean absolute correlation: {mean_abs_corr:.3f}")
print(f"Kept {len(kept_neurons)} / {len(best_interpretations)} neurons after fidelity filtering")


[annotate] cache hits: 0 / 32000; querying 32000 items


Scoring neuron interpretation fidelity (32 neurons × 200 examples):   0%|          | 0/32000 [00:00<?, ?it/s]

Mean absolute correlation: 0.394
Kept 22 / 32 neurons after fidelity filtering


## Step 7: Compute Controlled Logit Coefficients

Compute length-controlled logit coefficients for each SAE feature to understand their predictive importance.


In [18]:
selection_cfg = SelectionSettings(
    controls=["length_delta"],
    classification=True,
    standardize=True,
)

logit_coef_map = compute_controlled_logit_coefs(
    activ_train,
    activ_val,
    train_pred_df,
    val_pred_df,
    train_pred_mask,
    val_pred_mask,
    dataset_cfg,
    selection_cfg,
)

print(f"Computed logit coefficients for {len(logit_coef_map)} neurons")


Computed logit coefficients for 32 neurons


## Step 8: Build Feature Table

Compile all results into a feature table with interpretations, abbreviations, fidelity scores, and coefficients.


In [19]:
# Abbreviate interpretations
interpretations_list = [best_interpretations.get(int(idx)) for idx in range(activ_train.shape[1])]

def abbreviate_interpretations(interpretations, abbreviator_model, n_workers=8):
    def _fn(text):
        stripped = text.strip()
        if not stripped:
            return ""
        return abbreviate_concept(stripped, model=abbreviator_model)
    return parallel_apply([interp or "" for interp in interpretations], _fn, n_workers=n_workers, desc="Abbreviating")

abbreviations = abbreviate_interpretations(interpretations_list, interpret_cfg.abbreviator_model)

# Measure redundancy
redundancy, redundancy_neighbors, n_components = measure_interpretation_redundancy(
    [interp for interp in interpretations_list if interp],
    similarity_threshold=0.7
)

# Compute firing rates
neuron_indices = np.arange(activ_train.shape[1])
firing_rates = {
    int(idx): float((activ_train[:, idx] != 0).mean()) for idx in neuron_indices
}

# Build feature table
feature_rows = []
for i, idx in enumerate(neuron_indices):
    interp = best_interpretations.get(int(idx))
    metrics = best_metrics.get(int(idx), {})
    feature_rows.append({
        "neuron_idx": int(idx),
        "interpretation": interp,
        "abbreviated_interpretation": abbreviations[i] if i < len(abbreviations) else None,
        "prevalence": firing_rates.get(int(idx)),
        "correlation": metrics.get("correlation"),
        "p_value": metrics.get("p_value"),
        "kept": int(idx) in kept_neurons,
        "semantic_redundancy": float(redundancy[i]) if i < len(redundancy) else np.nan,
        "semantic_neighbor": redundancy_neighbors[i] if i < len(redundancy_neighbors) else None,
        "length_controlled_logit_coef": logit_coef_map.get(int(idx)),
    })

feature_table = pd.DataFrame(feature_rows)

print(f"Feature table created with {len(feature_table)} neurons")


Abbreviating:   0%|          | 0/32 [00:00<?, ?it/s]

Feature table created with 32 neurons


## Results Summary

Display AUC metrics and feature table with top features by coefficient magnitude.


In [20]:
print("=" * 60)
print("AUC METRICS")
print("=" * 60)
print(f"Embedding AUC - Train: {embedding_train_auc:.4f}, Val: {embedding_val_auc:.4f}")
print(f"SAE AUC - Train: {sae_train_auc:.4f}, Val: {sae_val_auc:.4f}")
print(f"Mean absolute fidelity correlation: {mean_abs_corr:.4f}")
print(f"Kept neurons: {len(kept_neurons)} / {len(neuron_indices)}")
print(f"Semantic components (number of semantically distinct feature interpretations, more means less redundancy): {n_components}")

print("\n" + "=" * 60)
print("TOP FEATURES BY LOGIT COEFFICIENT")
print("=" * 60)
display_cols = [
    "interpretation",
    "abbreviated_interpretation",
    "length_controlled_logit_coef",
    "correlation",
    "p_value",
    "prevalence",
]

sorted_features = (
    feature_table.dropna(subset=["length_controlled_logit_coef"])
    .sort_values("length_controlled_logit_coef", ascending=False)
)

with pd.option_context("display.max_colwidth", None):
    display(sorted_features[display_cols].round(3))

AUC METRICS
Embedding AUC - Train: 0.7359, Val: 0.6728
SAE AUC - Train: 0.6679, Val: 0.6704
Mean absolute fidelity correlation: 0.3943
Kept neurons: 22 / 32
Semantic components (number of semantically distinct feature interpretations, more means less redundancy): 25

TOP FEATURES BY LOGIT COEFFICIENT


,interpretation,abbreviated_interpretation,length_controlled_logit_coef,correlation,p_value,prevalence
3,"uses inclusive, values-oriented language highlighting empathy, respect, and equality","uses inclusive, values-based language emphasizing empathy, respect, equality",0.169,0.666,0.000,0.246
25,"provides on-topic, actionable suggestions","gives relevant, actionable suggestions",0.118,0.235,0.001,0.127
2,directly answers the user's prompt with substantive content,directly answers prompt with substantive content,0.118,0.638,0.000,0.239
22,does not include unrelated or extraneous content,excludes unrelated/extraneous content,0.097,0.263,0.000,0.108
30,"provides neutral, non-committal responses instead of taking a definitive stance","gives neutral, noncommittal responses rather than definitive stances",0.086,0.202,0.004,0.108
15,"offers generic, empathetic guidance rather than specific, detailed advice or resources","offers general empathetic guidance, not specific detailed advice or resources",0.074,0.260,0.000,0.118
20,"provides a detailed, on-topic answer instead of refusing, deflecting, or giving a terse reply","gives detailed, on-topic answers rather than refusing, deflecting, or responding tersely",0.058,0.430,0.000,0.024
5,expresses neutrality and emphasizes complexity instead of taking a firm stance,"expresses neutrality, emphasizes complexity over firm stance",0.054,0.646,0.000,0.240
0,is verbose and multi-paragraph,"verbose, multi-paragraph",0.048,0.809,0.000,0.267
31,"provides neutral, informational answers without expressing personal opinions or emotions","provides neutral, informational answers without opinions or emotions",0.032,0.170,0.016,0.131
